### 복습
- data 폴더 안에 가전 폴더의 모든 데이터 셋을 로드하여 하나의 데이터프레임으로 생성
- 감성(GeneralPolarity) 컬럼의 데이터를 -1,0,1 에서 0,1,2 로 변환
- RawText의 데이터에서 텍스트 정규화, 중복 데이터 제거, 글자 수가 1 이하인 데이터 제거
- 감성이 결측치인 데이터를 따로 저장
- 원본의 데이터에서는 결측치를 제거
- GenralPolarity 컬럼의 이름을 labels로 변경
- RawText, labels 컬럼만 유지
- train, test 데이터셋 분할(8:2)
- SBERT 모델을 생성 ('jhgan/ko-sroberta-multitask')
- Datset에서 SBERT 모델을 이용하여 임베딩 (생성자 함수에서 일괄 처리)
- Dataloader를 이용하여 batch_size = 128 배치 데이터 생성
- 다중 퍼셉트론 모델을 생성하여 감성분석 (Linear -> ReLU -> Dropout -> Linear)
- 반복 학습의 횟수 20
- 검증 데이터를 이용하여 f1_score를 확인
- 결측치 데이터에서 랜덤하게 5개 데이터를 추출하여 예측값을 확인
- 딥러닝 모델이 아닌 SVC 모델을 이용하여 f1_score 확인

In [57]:
import os
import re
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sentence_transformers import SentenceTransformer
from sklearn.svm import SVC

In [17]:
def normalize(text):
    text=re.sub(r'[^가-핳0-9a-zA-Z\s\.]',' ', str(text))
    text=re.sub(r'\s+',' ', text).strip()
    return text

In [ ]:
#os라이브러리를 이용하여 파일의 목록을 로드
file_path='../data/가전/'   #마지막에 /를 시용하는 이유는? 파일이름과 경로를 연결해서 사용할때 편리하기 위함
file_list=os.listdir(file_path)
file_list

In [19]:
df=pd.DataFrame()

for file in file_list:
    #file : 파일 이름
    data=pd.read_json(file_path + file)
    df=pd.concat([df, data], axis=0)
df.info()

<class 'pandas.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   str    
 2   Source           4056 non-null   str    
 3   Domain           4056 non-null   str    
 4   MainCategory     4056 non-null   str    
 5   ProductName      4056 non-null   str    
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(1), str(5)
memory usage: 3.9+ MB


In [20]:
#필요한 컬럼을 재외하고 나머지 컬럼을 삭제
df=df[['RawText', 'GeneralPolarity']]

In [21]:
#특정 컬럼의 이름을 변경 : 객체 안에 변수를 직접 수정(df.columns), rename() 함수를 이용
df.rename(columns = {
    'GeneralPolarity' : 'labels'
}, inplace=True)

In [22]:
#RawText 컬럼의 데이터를 텍스트 정규화
df['RawText']=df['RawText'].map(normalize)

In [23]:
#길이가 1 이하인 데이터는 제외
flag=df['RawText'].str.len() > 1
df=df.loc[flag,]

In [24]:
#RawText의 중복 데이터를 제거
df.drop_duplicates('RawText',inplace=True)

In [25]:
#labels의 결측치    가 존재 -> 결측치만 따로 저장
df_na=df.loc[df['labels'].isna(),]
df2=df.loc[~(df['labels'].isna()),]
df2.info()

<class 'pandas.DataFrame'>
Index: 3678 entries, 0 to 99
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  3678 non-null   str    
 1   labels   3678 non-null   float64
dtypes: float64(1), str(1)
memory usage: 2.9 MB


In [26]:
df_na.info()

<class 'pandas.DataFrame'>
Index: 378 entries, 13 to 88
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  378 non-null    str    
 1   labels   0 non-null      float64
dtypes: float64(1), str(1)
memory usage: 317.3 KB


In [27]:
df2['labels'].value_counts()

labels
 1.0    2220
 0.0     944
-1.0     514
Name: count, dtype: int64

In [28]:
#labels 컬럼의 데이터 타입을 int로 변경 -> 
#딥러닝 모델에서 분류 모델을 생성할 때 3개의 컬럼의 수치가 등록 -> 가장 높은 값을 가진 위치가 정답
#위치 값은 int, 위치는 0부터 시작
df2['labels']=df2['labels'].astype(int)

In [29]:
df2['labels']=df2['labels'].map(
    {
        -1 : 0,
        0 : 1,
        1 : 2
    }
)

In [30]:
# df2['labels'] + 1

In [31]:
df2['labels'].value_counts()    

labels
2    2220
1     944
0     514
Name: count, dtype: int64

In [36]:
#train, test 데이터셋 분할
train_df, test_df=train_test_split(
    df2, test_size=0.2, random_state=42, stratify=df2['labels']
)

train_df['labels'].value_counts()

labels
2    1776
1     755
0     411
Name: count, dtype: int64

In [37]:
model_name='jhgan/ko-sroberta-multitask'
sbert=SentenceTransformer(model_name)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5815.44it/s]


In [38]:
#최대 토큰 길이 제한
sbert.max_seq_length=128

In [42]:
class SBERTDataset(Dataset):
    def __init__(self, text, labels):
        #생성자 함수에서 임베딩 처리를 완료
        #객체 생성 시 데이터의 모든 임베딩 처리를 완료 -> class가 생성이 되는 과정에서는 시간이 걸리지만 딥러딩 반복 학습에서 시간이 단축 (메모리의 사용량은 증가)
        #대용량의 데이터를 이용해서 Dataset을 생성하는 경우에는 OOM(Out Of Memory)문제가 발생할 수 있다.
        with torch.inference_mode():
            self.emb=sbert.encode(      #encode() : 토큰화 -> 인코딩 -> 벡터화 (스케일링)
                text, convert_to_tensor=True, normalize_embeddings=True
            )
            self.labels=torch.tensor(labels,dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self,idx):
        return self.emb[idx], self.labels[idx]

In [43]:
train_ds=SBERTDataset(train_df['RawText'].tolist(), train_df['labels'].tolist())
test_ds=SBERTDataset(test_df['RawText'].tolist(), test_df['labels'].tolist())

In [44]:
train_dl=DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl=DataLoader(test_ds, batch_size=128, shuffle=True)

In [56]:
#다중 퍼셉트론 구조의 분류 모델을 선언
class MLPModel(nn.Module):
	def __init__(self, input_dim, hidden_dim=256, num_classes=2, dropout=0.5):
	    super().__init__()
		self.net=nn.Sequential(
			nn.Linear(input_dim, hidden_dim),
		    nn.ReLU(),
		    nn.Dropout(dropout),
		    nn.Linear(hidden_dim, num_classes)
		    )
	#순전파 함수
	def forward(self,X):
		result=self.net(X)
		return result

In [47]:
#sbert 모델에서 출력 차원의 개수를 확인
in_dim=sbert.get_sentence_embedding_dimension()
in_dim

C:\Users\user\AppData\Local\Temp\ipykernel_17368\2439650241.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  in_dim=sbert.get_sentence_embedding_dimension()


768

In [51]:
#모델 생성
model=MLPModel(in_dim, num_classes=3)
#손실 함수
#데이터의 불균형 존재
	#샘플링 기법, ML 모델에서는 class_weight 매개변수를 이용해서 밸런스를 맞춰주는 방법
	#DL 모델에서는 손실함수에서 클래스별 패널티를 저장 (weight 매개변수)
		#[0, 1, 2] -> 데이터가 적을수록 높은 변화
		#5.0 ~ 10.0 : 데이터의 비율이 굉장히 적은 경우
		#2.0 ~ 4.0 : 데이터의 비율이 적은 경우
		#1.0 : 가장 많은 데이터
criterion=nn.CrossEntropyLoss(weight=torch.tensor([5.0, 3.0, 1.0]))
#옵티마이저
optimizer=optim.AdamW(model.parameters(), lr=2e-04)


In [52]:
#학습 모드 전환
model.train()

for epoch in range(20):
	total=0.0
	for X, y in train_dl:
		#기울기 초기화
		optimizer.zero_grad()
		#예측값 생성
		logits=model(X)
		#
		loss=criterion(logits,y)
		#
		loss.backward()
		#
		optimizer.step()

		total +=loss.item() * X.size(0)
	if (epoch+1)%5 == 0:
		print(f'Epoch : {epoch+1} , loss : {round(total/len(train_df),4)}')


Epoch : 5 , loss : 0.9161
Epoch : 10 , loss : 0.7376
Epoch : 15 , loss : 0.6695
Epoch : 20 , loss : 0.6472


In [53]:
#예측 값 생성
model.eval()

y_true, y_pred=[], []

with torch.inference_mode():
	for X, y in test_dl:
		logits=model(X)
		pred=logits.argmax(dim=-1).tolist()
		y_true.extend(y.tolist())
		y_pred+=pred

In [58]:
print(accuracy_score(y_true, y_pred))
print(f1_score(y_true, y_pred, average='macro'))

0.6698369565217391
0.6676271417739521


In [60]:
#결측치 데이터중 랜덤하게 5개 추출
samples=df_na['RawText'].sample(5).tolist()
samples

['머리가 미친 듯한 곱슬머리라 매직기는 저한테 생명같아요. 대가 가능한 매직 고데기가 너무 갖고 싶었는데 이 상품은 디자인도 너무 스타일리쉬 서 좋아요. 사이즈도 작아서 가방에 넣고 다녀도 부담이 없어서 좋아요. 온도 조절도 5단계나 가능하다 보니 앞머리랑 옆머리 할 때 다르게 사용할 수 있다는 점이 말할 것도 없이 좋네요. 머리 손상도 많아서 모질이 안 좋은데 열판도 손상을 덜어준다고 하니 믿고 안심하고 사용하고 있습니다. 처음엔 디자인이 너무 예뻐서 사게 됐는데 설명서도 읽어보고 사용 보니 제품도 질이 아주 좋은 것같아 만족도가 사용할 수록 높아지네요',
 '와플 기계로 별거 다 먹는 방송 보고나도 하나 살까 서 구입 어요 이것 저것 비교 보고 와플 메이커로 주문 는데탁월한 초이스 였어요원래 가전은 거 선 하는데 왠지 레드가 강렬 서 구입 는데색상 끈하고 맘에 듭니다 꼭 와플이 아니라도 식빵은 기본이고 떡도 넣어서 눌러보고요 식감이 재밌네요 특이한 점은 양쪽으로 분리할 수 있어 씻을때도 편하고요그냥 닦을 때도 편하고 한쪽에는 샌드위치 빵 굽고 한쪽에는 와플 굽기가 동시에 할 수 있어 진짜 편하고 좋은 거 같아요완전 도 상품이네요',
 '에어 프라이어가 있기는 한데 용량이 작아서 감자튀김이라도 한 번 하면 모자라더라구요.먹다가 또 꺼내서 돌리고 하면 시간도 많이 걸렸는데 이 제품이 용량이 커서 바로 주문 습니다. 많은 재료가 들어가서 좋아요 겉에 재질도 좋고 스테인레스로 되어 있어 세척하고 청소 관리하기도 아주 좋네요.그리고 은색 색감이 주방에 싱크대 위에 올려 놓아도 아주 고급스러워 보입니다.냉장고 색상이랑 비슷 서 더 잘 어울린다고 느끼나봐요.그리고 레시피 없이도 다이얼로 편리하게 에어프라이용과 오븐용을 구별 서 할 수 있어서 멀티 기능을 가졌지만 두 개의 제품을 따로 쓰는 것같은 과도 주는게 아주 똑똑하고 만족스러워요.',
 '이 제품은 저를 위한 겁니다 머리를 말리는 거야 대충 물기야 사라져라 하는데... 항상 앞머리가 문제더라고요.앞머리가 제대로 

In [61]:
@torch.no_grad()
def predict_review(text, batch_size=128):
	#samples 데이터에서 임베딩 처리
	sbert.eval()
	model.eval()

	result=[]
	#임의로 배치 사이즈 생성
	for idx in range(0, len(text), batch_size):
		#배치 구간을 생성
		batch_text=text[idx : idx + batch_size]

		#배치 구간을 encode() 대입
		embs=sbert.encode(
			batch_text, convert_to_tensor=True, normalize_embeddings=True
		)
		#예측값 생성
		logits=model(embs)
		#예측 값을 이용하여 확률로 변환
		probs=logits.softmax(dim=-1)
		#예측 위치 계산
		preds=probs.argmax(dim=-1).tolist()
		#예측값을 변환하는 dict
		label_to_id={
			0 : '부정',
			1 : '중립',
			2 : '긍정'
		}
		
		for idx2, pred in enumerate(preds):
			#idx : 전체 문장에서 배치의 시작 위치
			#idx2 ; 베치 안에서의 초기화된 위치 값
			review=text[idx+idx2]	#idx+idx2 : 전체 문장에서 해당 문장의 위치	
			#예측 확률
			prob=float(probs[idx2, pred])
			#예측 값에 따른 id로 변환
			label=label_to_id[pred]
			
			result.append(
				{
					'text' : review,
					'prob' : prob,
					'label' : label
				}
			)
	return result

pd.DataFrame(predict_review(samples))

,text,prob,label
0,머리가 미친 듯한 곱슬머리라 매직기는 저한테 생명같아요. 대가 가능한 매직 고데기가...,0.651944,긍정
1,와플 기계로 별거 다 먹는 방송 보고나도 하나 살까 서 구입 어요 이것 저것 비교 ...,0.512524,중립
2,에어 프라이어가 있기는 한데 용량이 작아서 감자튀김이라도 한 번 하면 모자라더라구요...,0.565023,중립
3,이 제품은 저를 위한 겁니다 머리를 말리는 거야 대충 물기야 사라져라 하는데... ...,0.473249,중립
4,애벌빨래가 되는 용으로 구매하려고 를 뒤져보고 검색하고 찾다보니 선택하게 된 제품입...,0.613740,긍정


In [62]:
svc=SVC(random_state=42)

In [63]:
train_ds[0:len(train_ds)]

(tensor([[-0.0305, -0.0212, -0.0011,  ...,  0.0317, -0.0233, -0.0427],
         [-0.0049, -0.0224,  0.0087,  ..., -0.0229,  0.0388, -0.0709],
         [-0.0413, -0.0586,  0.0038,  ...,  0.0278,  0.0280, -0.0094],
         ...,
         [-0.0164, -0.0241, -0.0033,  ...,  0.0638, -0.0231, -0.0767],
         [-0.0209, -0.0299,  0.0007,  ...,  0.0305,  0.0120, -0.0863],
         [-0.0386, -0.0581,  0.0403,  ...,  0.0183,  0.0565, -0.0426]]),
 tensor([2, 2, 1,  ..., 1, 2, 1]))

In [64]:
X_train,y_train=train_ds[0:len(train_ds)]
X_test,y_test=test_ds[0:len(test_ds)]

In [65]:
import numpy as np

In [66]:
svc.fit(np.array(X_train), np.array(y_train))

C:\Users\user\AppData\Local\Temp\ipykernel_17368\1507869689.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  svc.fit(np.array(X_train), np.array(y_train))


,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forprobability estimates. Ignored when `probability` is False.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None


In [67]:
pred=svc.predict(np.array(X_test))

C:\Users\user\AppData\Local\Temp\ipykernel_17368\1085844236.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  pred=svc.predict(np.array(X_test))


In [68]:
f1_score(pred, np.array(y_test),average='macro')

C:\Users\user\AppData\Local\Temp\ipykernel_17368\78998477.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  f1_score(pred, np.array(y_test),average='macro')


0.6847829970584246